In [1]:
from datetime import datetime
import pandas as pd

from diive.core.io.files import load_parquet, save_parquet
from diive.pkgs.gapfilling.xgboost_ts import XGBoostTS

In [2]:
df = load_parquet(filepath="17.3_CH-CHA_meteo10_2005-2024.parquet")
df

Loaded .parquet file 17.3_CH-CHA_meteo10_2005-2024.parquet (0.745 seconds).
    --> Detected time resolution of <30 * Minutes> / 30min 


,LW_IN_T1_2_1,PA_GF1_0.9_1,FLAG_PA_GF1_0.9_1_ISFILLED,PPFD_IN_T1_2_2,FLAG_PPFD_IN_T1_2_2_ISFILLED,VPD_T1_2_1,...,TS_GF1_0.04_1_gfXG,FLAG_TS_GF1_0.04_1_gfXG_ISFILLED,TS_GF1_0.15_1_gfXG,FLAG_TS_GF1_0.15_1_gfXG_ISFILLED,TS_GF1_0.4_1_gfXG,FLAG_TS_GF1_0.4_1_gfXG_ISFILLED
TIMESTAMP_MIDDLE,,,,,,,,,,,,,
2005-01-01 00:15:00,NaN,978.100000,1.0,0.0,0,0.099893,...,1.014525,1,2.907254,1,4.007686,1
2005-01-01 00:45:00,NaN,977.933333,1.0,0.0,0,0.097606,...,1.029936,1,2.907254,1,4.007686,1
2005-01-01 01:15:00,NaN,977.900000,1.0,0.0,0,0.091683,...,1.003078,1,2.903765,1,4.077782,1
2005-01-01 01:45:00,NaN,977.833333,1.0,0.0,0,0.071157,...,1.056877,1,2.903765,1,4.077782,1
2005-01-01 02:15:00,NaN,977.833333,1.0,0.0,0,0.058333,...,0.963062,1,2.932330,1,3.979915,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 21:45:00,304.613900,983.370890,NaN,0.0,0,0.000011,...,3.474346,0,4.437078,0,5.528727,0
2024-12-31 22:15:00,303.039890,983.052160,NaN,0.0,0,0.000011,...,3.428224,0,4.440415,0,5.521962,0
2024-12-31 22:45:00,302.093633,982.851140,NaN,0.0,0,0.000011,...,3.384733,0,4.443751,0,5.523991,0


# List of meteo variables

In [3]:
[print(c) for c in df.columns];

LW_IN_T1_2_1
PA_GF1_0.9_1
FLAG_PA_GF1_0.9_1_ISFILLED
PPFD_IN_T1_2_2
FLAG_PPFD_IN_T1_2_2_ISFILLED
VPD_T1_2_1
FLAG_VPD_T1_2_1_ISFILLED
SW_IN_T1_2_1
FLAG_SW_IN_T1_2_1_ISFILLED
TA_T1_2_1
FLAG_TA_T1_2_1_ISFILLED
RH_T1_2_1
FLAG_RH_T1_2_1_ISFILLED
PREC_RAIN_TOT_GF1_0.5_1
FLAG_PREC_RAIN_TOT_GF1_0.5_1_ISFILLED
SWC_GF1_0.05_1
SWC_GF1_0.15_1
SWC_GF1_0.75_1
TS_GF1_0.04_1
TS_GF1_0.15_1
TS_GF1_0.4_1
FLAG_PREC_RAIN_TOT_GF1_0.5_1_FLUXNET_ISFILLED
TIMESINCE_PREC_RAIN_TOT_GF1_0.5_1
TS_GF1_0.04_1_gfXG
FLAG_TS_GF1_0.04_1_gfXG_ISFILLED
TS_GF1_0.15_1_gfXG
FLAG_TS_GF1_0.15_1_gfXG_ISFILLED
TS_GF1_0.4_1_gfXG
FLAG_TS_GF1_0.4_1_gfXG_ISFILLED


# Stats

In [4]:
df['YEAR'] = df.index.year
r = df.groupby(df['YEAR']).agg(['mean', 'min', 'max'])
r

LW_IN_T1_2_1                         PA_GF1_0.9_1                           ... TS_GF1_0.4_1_gfXG                      FLAG_TS_GF1_0.4_1_gfXG_ISFILLED        
             mean         min         max         mean         min          max  ...              mean       min        max                            mean min max
YEAR                                                                             ...                                                                               
2005   313.975454  183.714477  408.134369   967.293296  943.433333   986.300000  ...         11.337275  1.981613  21.500950                        0.688813   0   1
2006   322.552220  194.135681  428.587158   966.978136  943.133333   985.133333  ...         11.265405  1.378100  21.740999                        0.003767   0   1
2007   319.351836  190.564819  428.511627   966.928428  939.900000   985.733333  ...         11.464346  3.496700  19.568001                        0.001142   0   1
2008   317.981805  195.482910  431.197479   965.769919  935.700000   988.666667  ...         10.852941  2.100000  19.441000                        0.000626   0   1
2009   322.691319  177.452698  450.135071   964.719762  929.166667   981.300000  ...         11.009709  2.539000  19.673000                        0.000856   0   1
2010   321.252809  188.583496  447.427185   963.093168  929.966667   980.600000  ...         10.997934  2.720900  21.628000                        0.026370   0   1
2011   321.595072  196.672394  452.027344   967.503621  934.333333   987.033333  ...         11.987387  2.745000  20.403000                        0.028653   0   1
2012   322.346812  162.974472  436.758179   966.258457  939.900000   984.566667  ...         11.548465  1.756000  20.605000                        0.017133   0   1
2013   321.527206  188.637634  429.210938   965.175403  938.933333   987.066667  ...         10.884896  2.639700  21.534000                        0.000342   0   1
2014   325.959922  183.454346  434.617004   964.062815  938.700000   983.300000  ...         11.896689  3.249800  20.238001                        0.000913   0   1
2015   322.346686  206.038223  436.109985   967.542529  926.800000   986.666667  ...         11.816261  2.524400  21.813999                        0.001941   0   1
2016   331.131278  195.191011  445.627448   970.823510  937.000000   996.663400  ...         11.927214  4.348000  21.198788                        0.131318   0   1
2017   328.619956  205.314700  438.321600   972.151073  937.815400   996.803600  ...         11.059409  2.296560  18.810770                        0.006678   0   1
2018   326.681934  176.715900  454.941700   969.371754  939.631400   991.259900  ...         11.866678  2.368998  21.408090                        0.138527   0   1
2019   320.894567  200.693800  439.717600   969.840996  935.627400  1004.485000  ...         11.606276  3.240576  19.773830                        0.007192   0   1
2020   320.404386  199.291100  446.894100   970.887839  935.608200   994.202400  ...         11.886455  4.165008  20.125170                        0.006831   0   1
2021   320.390416  180.719637  429.912117   970.662426  945.188160   989.604030  ...         11.483076  3.462481  20.298380                        0.003025   0   1
2022   322.739784  200.726750  436.978833   971.599926  948.162753   991.971040  ...         12.473499  3.988803  20.936820                        0.000285   0   1
2023   328.901475  190.795627  450.673410   969.488699  930.964657   994.529460  ...         12.499827  3.401867  21.223760                        0.000285   0   1
2024   335.366847  182.633747  444.520093   969.631713  936.964837   991.404327  ...         13.064451  4.198476  22.356800                        0.001935   0   1

[20 rows x 87 columns]